# House Price Prediction — Random Forest
PE-CA2 examination case study. Dataset: [Kaggle House Price Dataset](https://www.kaggle.com/datasets/hosammhmdali/house-price-dataset).

Predict `median_house_value` (USD) for a California district using eight numerical features and `ocean_proximity`. These are district statistics, not individual house measurements.

**Before running:** open this notebook in Google Colab, select **Runtime → Change runtime type → T4 GPU → Save**, then **Runtime → Run all**. GPU availability depends on your Colab quota. Scikit-learn Random Forest runs on the CPU; selecting a GPU does not accelerate it. The runtime check below records the GPU allocation as examination evidence.

The notebook fits preprocessing only on the training set, evaluates once on a held-out 20% test set, and saves the complete fitted pipeline and its parameters. No results are recorded until the notebook actually runs.

In [ ]:
import sys, subprocess, json
from datetime import datetime, timezone
from pathlib import Path
import google.colab

runtime_started = datetime.now(timezone.utc).isoformat()
gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError("Select a GPU runtime in Colab, reconnect, and run again.")
print(gpu.stdout)
print("Python:", sys.version)


In [ ]:
%pip install -q numpy==2.2.6 pandas==2.2.3 scipy==1.15.3 scikit-learn==1.6.1 joblib==1.4.2


## 1. Load and validate the exact Kaggle dataset
The public download uses the supplied Kaggle dataset directly. If Kaggle requires authentication, download `housing.csv` from the dataset page and upload it using the fallback prompt.

In [ ]:
import hashlib, io, urllib.request, zipfile
import numpy as np
import pandas as pd
import sklearn, scipy, joblib

DATASET_URL = "https://www.kaggle.com/datasets/hosammhmdali/house-price-dataset"
EXPECTED_SHA256 = "8a3727f4cf54ac1a327f69b1d5b4db54c5834ea81c6e4efc0d163300022a685e"
try:
    url = "https://www.kaggle.com/api/v1/datasets/download/hosammhmdali/house-price-dataset"
    with urllib.request.urlopen(url, timeout=120) as response:
        archive = zipfile.ZipFile(io.BytesIO(response.read()))
    filename = next(n for n in archive.namelist() if n.lower().endswith("housing.csv"))
    raw = archive.read(filename)
except Exception as error:
    print("Download unavailable:", error)
    print("Upload housing.csv downloaded from the specified Kaggle page.")
    from google.colab import files
    uploaded = files.upload()
    raw = uploaded["housing.csv"]

checksum = hashlib.sha256(raw).hexdigest()
assert checksum == EXPECTED_SHA256, "Dataset differs from the validated version. Recheck its Kaggle source."
Path("housing.csv").write_bytes(raw)
df = pd.read_csv("housing.csv")
assert df.shape == (20640, 10)
assert df.duplicated().sum() == 0
assert df["median_house_value"].notna().all()
print("Shape:", df.shape)
print("Missing values:\n", df.isna().sum())
display(df.head())


## 2. Split, preprocess, and train
Use an 80/20 random split with seed 42. Fill missing numerical values using training medians and encode the category using one-hot encoding. Random Forest does not need feature scaling. Keep 200 trees, matching the simple approach taught in class.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

TARGET = "median_house_value"
FEATURES = [c for c in df.columns if c != TARGET]
NUMERIC_FEATURES = [c for c in FEATURES if c != "ocean_proximity"]
X = df[FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
    ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["ocean_proximity"])
])
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1))
])
model.fit(X_train, y_train)
print("Training rows:", len(X_train), "Test rows:", len(X_test))


## 3. Evaluate on unseen test rows
Report MAE and RMSE in USD and R² as a regression score. R² is not classification accuracy. Compare against a simple mean-price baseline. This historical dataset has capped house values and a random split does not measure performance on future markets.

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor

predictions = model.predict(X_test)
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
metrics = {
    "mae_usd": float(mean_absolute_error(y_test, predictions)),
    "rmse_usd": float(root_mean_squared_error(y_test, predictions)),
    "r2": float(r2_score(y_test, predictions)),
    "baseline_mae_usd": float(mean_absolute_error(y_test, baseline.predict(X_test))),
    "train_rows": len(X_train), "test_rows": len(X_test)
}
assert np.isfinite(list(metrics.values())).all()
assert metrics["mae_usd"] < metrics["baseline_mae_usd"], "Model did not beat the mean baseline."
print(json.dumps(metrics, indent=2))
display(pd.DataFrame({"actual_usd": y_test, "predicted_usd": predictions}).head(10))


## 4. Save the model, parameters, metrics, and execution evidence
The `.joblib` file includes the fitted preprocessing and all trained trees. JSON stores the Random Forest hyperparameters separately. Reopen the saved model and check that its predictions match before exporting. Keep the downloaded ZIP and save this executed notebook in Colab for examination evidence.

In [ ]:
import platform, shutil

output = Path("models")
output.mkdir(exist_ok=True)
model_path = output / "california_housing_rf_model.pkl"
joblib.dump(model, model_path, compress=3)
restored_model = joblib.load(model_path)
np.testing.assert_allclose(restored_model.predict(X_test.head(10)), predictions[:10])

parameters = model.named_steps["regressor"].get_params()
metadata = {
    "dataset_url": DATASET_URL, "dataset_sha256": checksum,
    "parameters": parameters, "test_metrics": metrics, "sklearn_version": sklearn.__version__,
    "target": TARGET, "currency": "USD", "features": FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "categories": model.named_steps["preprocessor"].named_transformers_["categorical"].categories_[0].tolist(),
    "test_size": 0.2, "random_state": 42,
    "training_environment": "Google Colab", "training_device": "CPU",
    "runtime_started_utc": runtime_started,
    "completed_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": platform.python_version(),
    "versions": {"numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__, "scipy": scipy.__version__, "joblib": joblib.__version__},
    "model_sha256": hashlib.sha256(model_path.read_bytes()).hexdigest(),
    "numeric_imputation_medians": model.named_steps["preprocessor"].named_transformers_["numeric"].statistics_.tolist()
}
example = X_test.iloc[0].astype(object).where(X_test.iloc[0].notna(), None).to_dict()
example_prediction = {"predicted_median_house_value_usd": float(restored_model.predict(X_test.iloc[[0]])[0])}
for name, value in [("parameters", parameters), ("metrics", metrics), ("california_housing_rf_metadata", metadata), ("example_request", example), ("example_prediction", example_prediction)]:
    (output / f"{name}.json").write_text(json.dumps(value, indent=2, allow_nan=False))
(output / "colab_gpu_runtime.txt").write_text(gpu.stdout)
(output / "model_requirements.txt").write_text("\n".join(f"{name}=={version}" for name, version in metadata["versions"].items()) + "\n")
print("Saved model reload verified.")
print(json.dumps(example_prediction, indent=2))
shutil.make_archive("colab_model_artifacts", "zip", root_dir=".", base_dir="models")
from google.colab import files
files.download("colab_model_artifacts.zip")
